# Cumberland Observed-Head Snapshot PEST Workflow

This notebook mirrors the first Cumberland PEST slice, but uses **real observed groundwater heads** instead of pseudo-targets from an earlier model run.

To keep the example light and focused on the current first-slice API, it still uses a **single steady-state stress period**. The observed targets come from one representative snapshot in the Cumberland observation workbook, using the period with the broadest well coverage.

The workflow shows:

- the same pre-development Cumberland geometry and top-surface adjustments
- the same current `K` and drain GIS inputs
- real observed well heads loaded from `calib_observations.xlsx`
- a `PestProject` with pilot-point `K` and drain conductance multipliers

This is a practical bridge between the synthetic “recover the model” case and a fuller transient calibration workflow.

## Imports and package access

Load the core Python, GIS, and `simple_modflow` pieces used for the Cumberland model build and first-slice PEST setup.

In [ ]:
from pathlib import Path
from datetime import datetime
import pickle
import re
import subprocess
import sys
import flopy
import numpy as np
import json
import geopandas as gpd
import pandas as pd

project_root = Path.cwd().resolve()
src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

import simple_modflow as mf
from simple_modflow import read_shp_gpkg
from simple_modflow.modflow.mf6.simulation import DisvGrid, TemporalDiscretization
from simple_modflow.modflow.mf6.simulation.packages import (
    Drains,
    InitialConditions,
    KFlow,
    OutputControl,
    Recharge,
    Storage,
)

## Workspace and live Cumberland inputs

Define a clean artifact workspace for this run and point to the same live Cumberland mesh and GIS inputs used by the recent pre-development workflows. This notebook also points to the real observation workbook.

In [ ]:
artifact_root = Path(r"C:\Users\lukem\Python\Projects\simple_modflow\examples\mf6\artifacts")
run_family = "cumberland_observed_snapshot_pest_first_slice"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
workspace_root = artifact_root / f"{run_family}_{timestamp}"
index = 1
while workspace_root.exists():
    workspace_root = artifact_root / f"{run_family}_{timestamp}_{index:02d}"
    index += 1
workspace_root.mkdir(parents=True)

model_workspace = workspace_root / "model"
pest_workspace = workspace_root / "pest"
latest_pointer = artifact_root / f"{run_family}_latest.txt"
latest_pointer.write_text(str(workspace_root), encoding="utf-8")

run_record = {
    "run_family": run_family,
    "workspace_root": str(workspace_root),
    "pest_workspace": str(pest_workspace),
    "created_at": timestamp,
}
(workspace_root / "run_info.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")

print(f"Workspace root: {workspace_root}")
print(f"Model workspace parent: {model_workspace}")
print(f"PEST workspace: {pest_workspace}")
print(f"Latest-run pointer: {latest_pointer}")


In [ ]:

ks_v5 = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\ks_v5.gpkg")
drn_path = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\drn.gpkg")
idomain_path = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\idomain.gpkg")
vor_algo = Path(r"C:\Users\lukem\mf6\Cumberland general\cumb_v14c_existing_algomesh_v2.1.vor")
iheads_path = Path(r"C:\Users\lukem\mf6\cumb_v6z_botm14V\iheads.hds")
pit2 = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\pit_2_footprint.gpkg")
pit4 = Path(r"C:\Users\lukem\mf6\Cumberland general\Herrera\pit4.gpkg")
rej = Path(r"C:\Users\lukem\mf6\Cumberland general\Herrera\min_elevs_to_fix_inf_rej.gpkg")
mfr_polys = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\mfr.gpkg")
calib_locs = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\calibration_locs.gpkg")
calib_obs = Path(r"C:\Users\lukem\mf6\Cumberland general\Tables\calib_observations.xlsx")


## Load the existing mesh and adjust surfaces

Open the saved Cumberland Voronoi mesh and starting heads, rebuild the active-domain mask, and apply the same pre-development surface adjustments used in `model_pit_pre_excavation.py`.

In [ ]:
with open(iheads_path, "rb") as f:
    iheads = pickle.load(f)

with open(vor_algo, "rb") as f:
    vor = pickle.load(f)

vor.gdf_topbtm[0] = vor.gdf_topbtm[0].astype(float)
vor.gdf_topbtm[1] = vor.gdf_topbtm[1].astype(float)
reconciled = vor.reconcile_surfaces(min_sep=5, trigger_sep=5, which="top")
vor.gdf_topbtm[0] = reconciled[0].astype(float)
vor.gdf_topbtm[1] = reconciled[1].astype(float)

inactive_cells = vor.get_vor_cells_as_series(idomain_path).explode().tolist()
idomain = pd.Series(1 for _ in range(vor.ncpl))
idomain.iloc[inactive_cells] = 0

pit2_cells = vor.get_vor_cells_as_series(pit2)[0]
tb2 = vor.gdf_topbtm.loc[pit2_cells]
pit2_thickness = tb2.loc[:, 0] - tb2.loc[:, 1]
thin_cells = pit2_thickness[pit2_thickness < 25].index
vor.gdf_topbtm.loc[thin_cells, 0] = vor.gdf_topbtm.loc[thin_cells, 1] + 25

pit4_cells = vor.get_vor_cells_as_series(pit4)[0]
tb4 = vor.gdf_topbtm.loc[pit4_cells]
pit4_thickness = tb4.loc[:, 0] - tb4.loc[:, 1]
thin_cells = pit4_thickness[pit4_thickness < 20].index
vor.gdf_topbtm.loc[thin_cells, 0] = vor.gdf_topbtm.loc[thin_cells, 1] + 20

rej_cells = vor.get_vor_cells_as_series(rej, name_field="name")
rej_df = pd.concat([read_shp_gpkg(rej).set_index("name"), rej_cells], axis=1)
for name, row in rej_df.iterrows():
    cells_to_check = vor.gdf_topbtm.loc[row.cells, 0]
    low_cells = cells_to_check.loc[cells_to_check < row.thickness].index.to_list()
    vor.gdf_topbtm.loc[low_cells, 0] = row.thickness

liz_cells = vor.get_vor_cells_as_series(read_shp_gpkg(mfr_polys).loc[8, "geometry"])[0]
lizmfr_cells = [cell for cell in liz_cells if cell not in inactive_cells]
vor.gdf_topbtm.loc[lizmfr_cells, 0] += 10

reconciled = vor.reconcile_surfaces(min_sep=5, trigger_sep=5, which="top")
vor.gdf_topbtm[0] = reconciled[0].astype(float)
vor.gdf_topbtm[1] = reconciled[1].astype(float)

top = vor.gdf_topbtm[0].values
botm = vor.gdf_topbtm[1].values

vor.ncpl, int(idomain.sum())

## Build and run the one-period pre-development model

Build the same simplified one-period Cumberland model used in the synthetic notebook: one steady-state stress period with the current `K` and drain GIS inputs, steady recharge, and the adjusted pre-development mesh.

In [ ]:
model = mf.SimulationBase(
    name="cpre_obs_ss",
    nper=1,
    vor=vor,
    mf_folder_path=model_workspace,
)

DisvGrid(
    vor=vor,
    model=model,
    nlay=1,
    top=top,
    bottom=botm,
    idomain=idomain,
)
TemporalDiscretization(model=model, period_data=[[1.0, 1, 1.0]])
OutputControl(model=model)

k_array = mf.KFromVector(
    model=model,
    vor=vor,
    shp_gpkg=ks_v5,
    uid="name",
    crs=2926,
    idomain_path=idomain_path,
).from_vector(nlay=1, defaults=[10.0])
KFlow(model=model, k=k_array[0], k33_vert=(k_array[0] / 10.0), save_specific_discharge=False)

InitialConditions(model=model, vor=vor, strt=[iheads])
Storage(
    model=model,
    specific_yield=0.2,
    specific_storage=1e-4,
    sto_steady={0: True},
    sto_transient={},
)

rch_dict = {
    0: [[(0, int(cell)), 0.007] for cell in range(vor.ncpl) if idomain.iloc[cell] == 1]
}
Recharge(model=model, vor=vor, rch_dict=rch_dict)

drn_fields = {
    "name": "name",
    "height_over_btm": "height",
    "conductance": "cond",
    "layer": "layer",
    "min_elev": "min_elev",
}
drn_spd = mf.DRNFromVector(
    model=model,
    vor=vor,
    shp_gpkg=drn_path,
    uid="name",
    crs=2926,
    idomain_path=idomain_path,
).from_vector(fields=drn_fields, edges_only=False, top_drain=True)
Drains(model=model, stress_period_data=drn_spd)

success, _ = model.run_simulation()
success

## Build a real observed-head snapshot

Read the Cumberland observation workbook, find the stress period with the broadest well coverage, and convert that one row into a `HeadTargets` object. This keeps the example compatible with the one-period first-slice workflow while still using real field observations.

In [ ]:
obs_frame = pd.read_excel(calib_obs)
well_columns = [column for column in obs_frame.columns if column not in {"per", "Deep Lake"}]
obs_frame["n_obs"] = obs_frame[well_columns].notna().sum(axis=1)
snapshot_period = int(obs_frame.sort_values("n_obs", ascending=False).iloc[0]["per"])
snapshot = obs_frame.loc[obs_frame["per"] == snapshot_period].iloc[0]
active_wells = [name for name in well_columns if pd.notna(snapshot[name])]

locs = gpd.read_file(calib_locs)
locs = locs[locs["ExploName"].astype(str).isin(active_wells)].copy()
locs["name"] = locs["ExploName"].astype(str)
locs["layer_num"] = 0
locs["weight"] = 1.0

values = pd.DataFrame([
    {"per": 0, **{name: float(snapshot[name]) for name in active_wells}}
])

targets = mf.HeadTargets(
    locations=locs[["name", "layer_num", "weight", "geometry"]],
    values=values,
    name_column="name",
    layer_column="layer_num",
    time_column="per",
)

print(f"Using real observed heads from workbook period {snapshot_period} with {len(active_wells)} wells.")
targets.summary()

## Build the PEST workspace against real observed heads

Create the same first-slice `PestProject` as before, but now assign actual observed head values to the PEST observations instead of pseudo-targets from an earlier model output table.

In [ ]:
raw_drn = gpd.read_file(drn_path)
raw_drn["par_name"] = raw_drn["name"].astype(str).str.lower().map(
    lambda s: re.sub(r"[^a-z0-9]+", "_", s).strip("_")
)
param_drn = workspace_root / "drn_param_source.gpkg"
if param_drn.exists():
    param_drn.unlink()
raw_drn.to_file(param_drn, driver="GPKG")

pest = mf.PestProject(
    model=model,
    name="cpre_obs_ss_pest",
    workspace=pest_workspace,
    start_datetime="2024-01-01",
)

pest.add_parameter(
    mf.KPilotPointParameter(
        name="hk",
        source=mf.VectorParameterSource(
            path=ks_v5,
            value_column="k",
            feature_id_column="name",
            zone_column="name",
            layer_column="layer",
            crs=2926,
        ),
        bounds=(0.25, 4.0),
        bounds_mode="multiplier",
        transform="log",
        pp_spacing=2500.0,
        geostruct=mf.ExpGeoStruct(range=4000.0, transform="log"),
    )
)

pest.add_parameter(
    mf.DrainConductanceParameter(
        name="drn_cond",
        source=mf.VectorParameterSource(
            path=param_drn,
            value_column="cond",
            feature_id_column="par_name",
            layer_column="layer",
            crs=2926,
        ),
        bounds=(0.25, 4.0),
        bounds_mode="multiplier",
        transform="log",
    )
)

pest.add_observation(mf.HeadTargetObservationSpec(targets=targets))
pst = pest.build_pst("cpre_obs_ss.pst")

pst.npar_adj, pst.nobs

## Review the active observed targets

Look at the actual observed heads that were assigned to the snapshot calibration problem, along with the simulated values and residuals from the baseline model.

In [ ]:
comparison = targets.compare(model)
comparison.loc[:, ["name", "head_target", "sim_head", "residual"]].sort_values("name").reset_index(drop=True)

## Quick residual summary

This gives a fast sense of how the baseline one-period pre-development model lines up with the chosen real observed-head snapshot before any calibration adjustments are made.

In [ ]:
targets.stats(model)

## Run `pestpp-glm`

This notebook now has everything needed to launch a small PEST++ calibration. Start with a low `noptmax` so you can verify the workflow and inspect the results before trying a longer solve.

In [ ]:
pst.control_data.noptmax = 1
npst_path = pest_workspace / "cpre_obs_ss.pst"
pst.write(npst_path)

pestpp_glm = "pestpp-glm"
print(f"Running PEST++ in: {pest_workspace}")
print(f"Control file: {npst_path.name}")

proc = subprocess.Popen(
    [pestpp_glm, npst_path.name],
    cwd=pest_workspace,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, proc.args)


## Review the PEST++ outputs

After the run finishes, inspect the key output files written by `pestpp-glm`. The control file, record file, residuals, and optimized parameter files are the first things to look at.

In [ ]:
sorted(path.name for path in pest_workspace.iterdir() if path.suffix.lower() in {".pst", ".rec", ".rei", ".par", ".csv", ".phi", ".jcb"})

## Reopen and evaluate a completed PEST run with the built-in API

Use this section after a finished calibration if you want to inspect the completed run without launching `pestpp-glm` again. The notebook logic here now uses the built-in `simple_modflow` reopen/evaluate API instead of notebook-only helper functions.


In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import simple_modflow as mf

DEFAULT_CALIB_LOCS = Path(r"C:\Users\lukem\mf6\Cumberland general\Boundaries\calibration_locs.gpkg")
DEFAULT_CALIB_OBS = Path(r"C:\Users\lukem\mf6\Cumberland general\Tables\calib_observations.xlsx")


def build_snapshot_targets_from_sources(calib_locs=DEFAULT_CALIB_LOCS, calib_obs=DEFAULT_CALIB_OBS):
    obs_frame = pd.read_excel(calib_obs)
    well_columns = [column for column in obs_frame.columns if column not in {"per", "Deep Lake"}]
    obs_frame["n_obs"] = obs_frame[well_columns].notna().sum(axis=1)
    snapshot_period = int(obs_frame.sort_values("n_obs", ascending=False).iloc[0]["per"])
    snapshot = obs_frame.loc[obs_frame["per"] == snapshot_period].iloc[0]
    active_wells = [name for name in well_columns if pd.notna(snapshot[name])]

    locs = gpd.read_file(calib_locs)
    locs = locs[locs["ExploName"].astype(str).isin(active_wells)].copy()
    locs["name"] = locs["ExploName"].astype(str)
    locs["layer_num"] = 0
    locs["weight"] = 1.0

    values = pd.DataFrame([{"per": 0, **{name: float(snapshot[name]) for name in active_wells}}])

    targets = mf.HeadTargets(
        locations=locs[["name", "layer_num", "weight", "geometry"]],
        values=values,
        name_column="name",
        layer_column="layer_num",
        time_column="per",
    )
    return targets, snapshot_period, active_wells


### Point to the completed run directory

Set this to either the completed artifact folder or the generated `pest` folder from a finished run. `mf.open_pest_run(...)` discovers the paired baseline model workspace, control file, and final parameter file automatically.


In [ ]:
completed_run_dir = Path(r"C:\Users\lukem\Python\Projects\simple_modflow\examples\mf6\artifacts\cumberland_observed_snapshot_pest_first_slice_01")

pest_run = mf.open_pest_run(completed_run_dir)
targets_post, snapshot_period_post, active_wells_post = build_snapshot_targets_from_sources()
run_summary = pest_run.summary()
display(run_summary)

if not pest_run.has_final_parameters:
    raise FileNotFoundError(
        "No final .par file was found in this PEST workspace. Pick a completed run that finished successfully."
    )

baseline_model = pest_run.load_baseline_model()
completed_model = pest_run.load_calibrated_model()
k_gdf = pest_run.k_geodata()

print(type(baseline_model))
print(type(completed_model))
print(f"Using real observed heads from workbook period {snapshot_period_post} with {len(active_wells_post)} wells")
print(f"Baseline workspace: {pest_run.baseline_workspace}")
print(f"PEST workspace: {pest_run.pest_workspace}")


### Plot the final `K` field and keep the reopened calibrated model for exploration

`completed_model` is a file-backed `LoadedMf6Run`, which is a subclass of `SimulationBase`. The built-in `mf.open_pest_run(...)` workflow reapplies the finished PEST parameter values in memory, so you can use the regular `simple_modflow` exploration APIs on the reopened calibrated model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
k_gdf.plot(column="k_final", ax=axes[0], legend=True)
axes[0].set_title("Final K")
axes[0].set_axis_off()

k_gdf.plot(column="k_ratio", ax=axes[1], legend=True)
axes[1].set_title("K final / K initial")
axes[1].set_axis_off()

fig


## Compare residuals before and after calibration

Use the built-in `PestRunResults` comparison helpers to see how much the first-slice calibration changed the observed-head residuals for the recreated Cumberland snapshot targets.


In [ ]:
residual_compare = pest_run.compare_head_targets(targets_post)
residual_compare.loc[:, [
    "name",
    "head_target",
    "sim_head_baseline",
    "sim_head_calibrated",
    "residual_baseline",
    "residual_calibrated",
    "abs_residual_improvement",
]].sort_values("abs_residual_improvement", ascending=False).reset_index(drop=True)


In [ ]:
pest_run.compare_head_target_stats(targets_post)
